In [2]:
import os
import json
import numpy as np
import json_repair
from tqdm.auto import tqdm

def load_jsonl_data(path):
    data = []
    with open(path) as reader:
        for row in reader:
            data.append(json.loads(row))
    return data

def write_jsonl_data(data, path):
    with open(path, "w") as writer:
        for row in data:
            writer.write(json.dumps(row, ensure_ascii=False) + "\n")

def _parse_json_response(text: str, keys: list) -> dict:
    import re
    
    try:
        try:
            result = json_repair.loads(text)
        except:
            match = re.search(r"```json\s*([\s\S]*?)\s*```", text)

            json_str = match.group(1).strip()
            result = json_repair.loads(json_str)

        for key in keys:
            if key not in result:
                result[key] = None

        return result

    except Exception as e:
        response = {k: None for k in keys}
        return response

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_eval_dataset(theme):
    root = '/inspire/hdd/project/socialsimulation/linjiayu-CZXS25120090/FYDUAN/exp_results/logs'

    base_dir = os.path.join(root, theme)
    paths = []

    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.startswith("sim_log"):
                paths.append(os.path.join(root, f))

    eval_for_ir = []
    eval_for_ic = []
    eval_for_pp = []
    eval_for_conv = []

    for p in paths:
        data = json.load(open(p))
        name = os.path.basename(os.path.dirname(p))

        sequence_id = data['event_sequence_info']['sequence_id']

        for i, x in enumerate(data['dialogue_log']):
            gold_intent = x['event']['intent']
            gold_sub_intents = x['event']['sub_intents']
            gold_preferences = x['user']['profile']['preferences_value']

            conv = ''
            pred_intents = ['']
            for j, turn in enumerate(x['dialogue']):
                if turn['role'] == 'user':
                    conv += f"[User]({turn['emotion']}) {turn['content']}\n"
                else:
                    conv += f"[Assistant] {turn['content']}\n"
                    
                    pre_intent = turn['pre_intent']
                    if pre_intent:
                        pred_intents.append(pre_intent)
                    eval_for_ir.append({
                        'id': '_'.join([name, str(i), str(j)]),
                        'sequence_id': sequence_id,
                        'user_id': x['user']['profile']['user_id'],
                        'gold_intent': gold_intent,
                        'gold_sub_intents': gold_sub_intents,
                        'pre_intent': pre_intent if pre_intent else ''
                    })
            
            pred_preferences = x['pre_profile']

            eval_for_pp.append({
                'id': '_'.join([name, str(i)]),
                'sequence_id': sequence_id,
                'user_id': x['user']['profile']['user_id'],
                'gold_preferences': gold_preferences,
                'pred_preferences': pred_preferences
            })

            eval_for_conv.append({
                'id': '_'.join([name, str(i)]),
                'sequence_id': sequence_id,
                'user_id': x['user']['profile']['user_id'],
                'user_profile': x['user']['profile_str'],
                'user_preferences': gold_preferences,
                "dialogue_scene": x['event']['dialogue_scene'],
                "event": x["event"]["event"],
                'intent': gold_intent,
                'sub_intents': gold_sub_intents,
                'conv': conv,
                'pred_intents': pred_intents
            })
    
    return eval_for_ir, eval_for_ic, eval_for_pp, eval_for_conv

In [4]:
themes = [
    'main_user_Qwen3-32B_assistant_Qwen3-8B_total',
    # 'main_user_Qwen3-32B_assistant_Qwen3-14B_total',
    'main_user_Qwen3-32B_assistant_Qwen3-32B_total',
    # 'main_user_Qwen3-32B_assistant_gemma-3-12b-it_total',
    # 'main_user_Qwen3-32B_assistant_gemma-3-27b-it_total',
    # 'main_user_Qwen3-32B_assistant_gpt-oss-20b_total',
    # 'main_user_Qwen3-32B_assistant_gpt-oss-120b_total',
    # 'main_user_Qwen3-32B_assistant_Meta-Llama-3.1-8B-Instruct_total',
    # 'main_user_Qwen3-32B_assistant_Meta-Llama-3.1-70B-Instruct_total'
]

index = 940
for theme in themes:
    print(theme)
    eval_for_ir, eval_for_ic, eval_for_pp, eval_for_conv = get_eval_dataset(theme)
    # for index in range(10, len(eval_for_conv)):
        # if 'annoyance' in eval_for_conv[index]['conv']:
        # print(theme)
        # print(eval_for_ir[0]['gold_intent'])
        # print(eval_for_ir[0]['pre_intent'])
        # print(index)

    print(eval_for_conv[index]['user_profile'])
    print(eval_for_conv[index]['dialogue_scene'])
    print(eval_for_conv[index]['event'])
    print(eval_for_conv[index]['intent'])
    print(eval_for_conv[index]['sub_intents'])
    print(eval_for_conv[index]['conv'])
    print('----' * 10)
    # break

main_user_Qwen3-32B_assistant_Qwen3-8B_total
Male, Elderly (over 65 years old), White Alone, Married, no religious affiliation, usually resides in Cities. the income level is High Income, Working now. Personality traits include: Dishonest.、Untrusting. Preferences expressed in daily life and interactions include: There are almost no special requirements for the work environment; able to quickly adapt to various office conditions, paying more attention to the work content itself rather than the external environment.
The time is 2012-04-06 12:42:10, Friday
The location is Park
The weather condition is Partially cloudy, described as Partly cloudy throughout the day.. The average temperature is 5.3°C (high of 11.8°C, low of -1.2°C).
On a brisk, partly cloudy Friday just after noon in the city park, the 68‑year‑old man slips on an uneven section of the walking path and scrapes his knee. He initially brushes it off and keeps walking before admitting to his spouse that he stumbled.
He asks the

In [6]:
# Proactive inquiry

themes = [
        'main_user_Qwen3-32B_assistant_gemma-3-12b-it_total',
        'main_user_Qwen3-32B_assistant_gemma-3-27b-it_total',
        'main_user_Qwen3-32B_assistant_gpt-oss-20b_total',
        'main_user_Qwen3-32B_assistant_gpt-oss-120b_total',
        'main_user_Qwen3-32B_assistant_Meta-Llama-3.1-8B-Instruct_total',
        'main_user_Qwen3-32B_assistant_Meta-Llama-3.1-70B-Instruct_total',
        'main_user_Qwen3-32B_assistant_Qwen3-8B_total',
        'main_user_Qwen3-32B_assistant_Qwen3-14B_total',
        'main_user_Qwen3-32B_assistant_Qwen3-32B_total',
        'main_user_Qwen3-32B_assistant_deepseek-chat_total',
        'main_user_Qwen3-32B_assistant_gpt-4o_total',
        'main_user_Qwen3-32B_assistant_deepseek-reasoner_total',
        'main_user_Qwen3-32B_assistant_claude-sonnet-4-5-20250929_total',
        'main_user_Qwen3-32B_assistant_gpt-5_total',
]

for theme in themes:
    eval_for_ir, eval_for_ic, eval_for_pp, eval_for_conv = get_eval_dataset(theme)
    rs = []
    for conv in eval_for_conv:
        us = conv['conv'].split('[User]')[1:]
        num_q = len([x for x in us if x.strip().endswith('?')])
        rs.append(num_q * 1.0 / len(us))
    print(theme)
    print(f'{np.mean(rs) * 100:.1f}')

main_user_Qwen3-32B_assistant_gemma-3-12b-it_total
13.7
main_user_Qwen3-32B_assistant_gemma-3-27b-it_total
34.2
main_user_Qwen3-32B_assistant_gpt-oss-20b_total
2.0
main_user_Qwen3-32B_assistant_gpt-oss-120b_total
1.7
main_user_Qwen3-32B_assistant_Meta-Llama-3.1-8B-Instruct_total
12.6
main_user_Qwen3-32B_assistant_Meta-Llama-3.1-70B-Instruct_total
22.4
main_user_Qwen3-32B_assistant_Qwen3-8B_total
5.1
main_user_Qwen3-32B_assistant_Qwen3-14B_total
4.9
main_user_Qwen3-32B_assistant_Qwen3-32B_total
9.8
main_user_Qwen3-32B_assistant_deepseek-chat_total
20.9
main_user_Qwen3-32B_assistant_gpt-4o_total
16.5
main_user_Qwen3-32B_assistant_deepseek-reasoner_total
29.2
main_user_Qwen3-32B_assistant_claude-sonnet-4-5-20250929_total
51.2
main_user_Qwen3-32B_assistant_gpt-5_total
16.6


In [9]:
import editdistance
import numpy as np

def find_closest_str_match(text, candidates):
    """
    输入:
        text: 待匹配的字符串
        candidates: 字符串列表
    输出:
        与 text 最接近的字符串
    """
    if not candidates:
        return None
    
    for c in candidates:
        if text.lower() in c.lower() or c.lower() in text.lower():
            return c

    distances = [editdistance.eval(text.lower(), candidate.lower()) for candidate in candidates]
    
    min_index = distances.index(min(distances))
    
    return candidates[min_index]

In [13]:
# Rigid Reasoning
from collections import defaultdict

root = '/inspire/hdd/project/socialsimulation/linjiayu-CZXS25120090/FYDUAN/ai_assistant/evaluation/assistant_performance/'

evaluators = [
    'gpt_oss_120b',
    # 'llama3_70b',
    # 'qwen3_32b'
]

models = [
    'main_user_Qwen3-32B_assistant_gemma-3-12b-it_total',
    'main_user_Qwen3-32B_assistant_gemma-3-27b-it_total',
    'main_user_Qwen3-32B_assistant_gpt-oss-20b_total',
    'main_user_Qwen3-32B_assistant_gpt-oss-120b_total',
    'main_user_Qwen3-32B_assistant_Meta-Llama-3.1-8B-Instruct_total',
    'main_user_Qwen3-32B_assistant_Meta-Llama-3.1-70B-Instruct_total',
    'main_user_Qwen3-32B_assistant_Qwen3-8B_total',
    'main_user_Qwen3-32B_assistant_Qwen3-14B_total',
    'main_user_Qwen3-32B_assistant_Qwen3-32B_total',
    'main_user_Qwen3-32B_assistant_deepseek-chat_total',
    'main_user_Qwen3-32B_assistant_gpt-4o_total',
    'main_user_Qwen3-32B_assistant_deepseek-reasoner_total',
    'main_user_Qwen3-32B_assistant_claude-sonnet-4-5-20250929_total',
    'main_user_Qwen3-32B_assistant_gpt-5_total',
]
    

for model in models:
    rr_results = defaultdict(int)
    for evaluator in evaluators:
        sub_rr_results = load_jsonl_data(os.path.join(root, evaluator, model, 'rr_results.jsonl'))
        for x in sub_rr_results:
            output = _parse_json_response(x['output'], ['rating'])
            # print(output)
            try:
                rr_results[x['id']] += float(output['rigid_reasoning'])
            except:
                rr_results[x['id']] += 0
    
    print(model)
    print(f"RR score: {np.mean(list(rr_results.values())) * 100 / len(evaluators): .1f}")

main_user_Qwen3-32B_assistant_gemma-3-12b-it_total
RR score:  62.7
main_user_Qwen3-32B_assistant_gemma-3-27b-it_total
RR score:  40.1
main_user_Qwen3-32B_assistant_gpt-oss-20b_total
RR score:  23.4
main_user_Qwen3-32B_assistant_gpt-oss-120b_total
RR score:  16.3
main_user_Qwen3-32B_assistant_Meta-Llama-3.1-8B-Instruct_total
RR score:  64.2
main_user_Qwen3-32B_assistant_Meta-Llama-3.1-70B-Instruct_total
RR score:  51.2
main_user_Qwen3-32B_assistant_Qwen3-8B_total
RR score:  61.8
main_user_Qwen3-32B_assistant_Qwen3-14B_total
RR score:  52.6
main_user_Qwen3-32B_assistant_Qwen3-32B_total
RR score:  42.1
main_user_Qwen3-32B_assistant_deepseek-chat_total
RR score:  30.8
main_user_Qwen3-32B_assistant_gpt-4o_total
RR score:  37.9
main_user_Qwen3-32B_assistant_deepseek-reasoner_total
RR score:  23.5
main_user_Qwen3-32B_assistant_claude-sonnet-4-5-20250929_total
RR score:  13.5
main_user_Qwen3-32B_assistant_gpt-5_total
RR score:  11.2
